# SAR Oil Spill Detection — Training Notebook

## Before running:
1. Make sure `colab_preprocessing.ipynb` has been run at least once
2. Go to **Runtime → Change runtime type → T4 GPU**
3. Run all cells top to bottom

## What this notebook does:
- Copies pre-converted `.npy` files from Drive to Colab local disk (~20-25 GB, one-time per session)
- Trains U-Net on GPU reading from local disk (~0.03s per file vs ~0.33s from tif)
- Saves best model and results back to Drive

---
## Cell 1 — Verify GPU

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'Memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU. Go to Runtime -> Change runtime type -> T4 GPU')

---
## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

def gb(x): return x / (1024**3)
_, _, free = shutil.disk_usage('/content')
print(f'Colab local disk free: {gb(free):.1f} GB')

# Paths
RESULTS_DIR = '/content/drive/MyDrive/Geo_Spill_results'
NPY_DRIVE   = f'{RESULTS_DIR}/npy_cache'
STATS_DRIVE = f'{RESULTS_DIR}/train_stats.json'
SPLITS_DRIVE = f'{RESULTS_DIR}/splits.json'

import os
if not os.path.exists(NPY_DRIVE):
    raise FileNotFoundError(
        f'NPY cache not found at {NPY_DRIVE}\n'
        'Run colab_preprocessing.ipynb first.'
    )

n_img  = len([f for f in os.listdir(f'{NPY_DRIVE}/images') if f.endswith('.npy')])
n_mask = len([f for f in os.listdir(f'{NPY_DRIVE}/masks')  if f.endswith('.npy')])
print(f'NPY files on Drive — images: {n_img}, masks: {n_mask}')

---
## Cell 3 — Copy NPY Files from Drive to Local Disk

This is a one-time cost per session (~10-15 minutes for ~20GB).
After copying, every training batch reads from local SSD at full speed.

If this cell is interrupted, just re-run it — it skips files already copied.

In [ ]:
import os, shutil
from tqdm import tqdm

LOCAL_NPY = '/content/npy_cache'
os.makedirs(f'{LOCAL_NPY}/images', exist_ok=True)
os.makedirs(f'{LOCAL_NPY}/masks',  exist_ok=True)

def copy_folder(src_dir, dst_dir, desc):
    files = [f for f in os.listdir(src_dir) if f.endswith('.npy')]
    copied = skipped = 0
    for fname in tqdm(files, desc=desc):
        dst = os.path.join(dst_dir, fname)
        if os.path.exists(dst):
            skipped += 1
            continue
        shutil.copy(os.path.join(src_dir, fname), dst)
        copied += 1
    print(f'  {desc}: {copied} copied, {skipped} already existed')

copy_folder(f'{NPY_DRIVE}/images', f'{LOCAL_NPY}/images', 'Images')
copy_folder(f'{NPY_DRIVE}/masks',  f'{LOCAL_NPY}/masks',  'Masks')

# Verify
n_local_img  = len(os.listdir(f'{LOCAL_NPY}/images'))
n_local_mask = len(os.listdir(f'{LOCAL_NPY}/masks'))
print(f'\nLocal disk — images: {n_local_img}, masks: {n_local_mask}')

result = !du -sh /content/npy_cache
print(f'Total size on local disk: {result[0].split()[0]}')
print('Ready for training!')

---
## Cell 4 — Load Stats and Update Splits to Local Paths

Loads `train_stats.json` and `splits.json` from Drive,
then rewrites the paths in splits to point at local disk.

In [ ]:
import json, os

with open(STATS_DRIVE)  as f: stats  = json.load(f)
with open(SPLITS_DRIVE) as f: splits = json.load(f)

# Remap Drive npy paths -> local npy paths
NPY_DRIVE_PREFIX = NPY_DRIVE      # e.g. /content/drive/MyDrive/Geo_Spill_results/npy_cache
NPY_LOCAL_PREFIX = '/content/npy_cache'

def to_local(p):
    return p.replace(NPY_DRIVE_PREFIX, NPY_LOCAL_PREFIX)

splits['train'] = [to_local(p) for p in splits['train']]
splits['val']   = [to_local(p) for p in splits['val']]
splits['test']  = [to_local(p) for p in splits['test']]
splits['masks'] = {to_local(k): to_local(v) for k, v in splits['masks'].items()}

# Save locally for train.py
os.makedirs('/content/GeoSpill-AI/data', exist_ok=True)
with open('/content/GeoSpill-AI/data/splits.json', 'w')      as f: json.dump(splits, f)
with open('/content/GeoSpill-AI/data/train_stats.json', 'w') as f: json.dump(stats, f)

print(f'Stats loaded   — mean: {stats["mean"]}')
print(f'Split          — {len(splits["train"])} train | {len(splits["val"])} val | {len(splits["test"])} test')
print(f'Sample path    — {splits["train"][0]}')

# Verify the first file actually exists
sample = splits['train'][0]
print(f'File exists    — {os.path.exists(sample)}')

---
## Cell 5 — Clone GitHub Repository

In [ ]:
import os

GITHUB_URL = 'https://github.com/TigranBoyakhchyan/GeoSpill-AI'
REPO_NAME  = 'GeoSpill-AI'

if os.path.exists(f'/content/{REPO_NAME}'):
    %cd /content/{REPO_NAME}
    !git pull origin main
else:
    !git clone {GITHUB_URL}
    %cd /content/{REPO_NAME}

print(f'Working directory: {os.getcwd()}')

---
## Cell 6 — Install Dependencies

In [ ]:
!pip install -q segmentation-models-pytorch albumentations rasterio
import segmentation_models_pytorch as smp
import albumentations as A
print(f'smp: {smp.__version__} | albumentations: {A.__version__}')

---
## Cell 7 — Quick Loading Speed Test

Verify that `.npy` loading is fast before starting full training.

In [ ]:
import time, numpy as np, os

sample_path = splits['train'][0]

# Warm up
_ = np.load(sample_path)

# Time it
times = []
for _ in range(10):
    t = time.time()
    img = np.load(sample_path)
    times.append(time.time() - t)

print(f'NPY load time (avg of 10): {np.mean(times)*1000:.1f} ms')
print(f'Image shape: {img.shape}, dtype: {img.dtype}')
print(f'Value range: [{img.min():.3f}, {img.max():.3f}]')

if np.mean(times) < 0.1:
    print('Loading speed is good!')
else:
    print('Loading is slower than expected — check if files are on local disk')

---
## Cell 8 — Train

- Reads pre-normalized `.npy` files from local disk
- Random 256×256 crops on-the-fly
- `--crops_per_image 10` → 1200 × 10 = 12,000 samples per epoch
- Expected time: **3-5 minutes per epoch** on T4 GPU (much faster than before)

In [ ]:
!python src/train.py \
    --epochs 50 \
    --batch_size 16 \
    --lr 1e-4 \
    --model_type smp \
    --crops_per_image 10

---
## Cell 9 — Plot Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv('checkpoints/training_log.csv')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training Curves', fontsize=14, fontweight='bold')

for ax, metric, title in zip(axes, ['loss', 'iou'], ['Loss', 'IoU']):
    ax.plot(log['epoch'], log[f'train_{metric}'], label=f'Train {title}', color='steelblue')
    ax.plot(log['epoch'], log[f'val_{metric}'],   label=f'Val {title}',   color='orange')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('checkpoints/training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

best = log.loc[log['val_iou'].idxmax()]
print(f'Best epoch   : {int(best["epoch"])}')
print(f'Best val IoU : {best["val_iou"]:.4f}')
print(f'Best val Dice: {best["val_dice"]:.4f}')

---
## Cell 10 — Save Results to Drive

**Always run before closing the session.**
Colab deletes everything in `/content/` when the session ends.

In [ ]:
import shutil, os

SAVE_DIR = '/content/drive/MyDrive/Geo_Spill_results'
os.makedirs(SAVE_DIR, exist_ok=True)

for src, dst in {
    'checkpoints/best_model.pth':      f'{SAVE_DIR}/best_model.pth',
    'checkpoints/training_log.csv':    f'{SAVE_DIR}/training_log.csv',
    'checkpoints/training_curves.png': f'{SAVE_DIR}/training_curves.png',
}.items():
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Saved: {src} -> Drive')
    else:
        print(f'Missing: {src}')

---
## Cell 11 — Commit Results to GitHub

**Replace email and name.**

In [ ]:
import pandas as pd

log  = pd.read_csv('checkpoints/training_log.csv')
best = log.loc[log['val_iou'].idxmax()]

!git config user.email "tigran@example.com"
!git config user.name  "Tigran Boyakhchyan"

!git add checkpoints/training_log.csv
!git add checkpoints/training_curves.png
!git add data/train_stats.json

msg = f'Colab training: val IoU={best["val_iou"]:.4f}, Dice={best["val_dice"]:.4f} ({len(log)} epochs)'
!git commit -m "{msg}"
!git push origin main
print(f'Pushed: {msg}')